# 02 · Previsões dos Big 6

Modelo **Dixon-Coles** (Poisson com correção para placares baixos e decaimento no tempo). Explicação completa em `src/plbig6/model.py`.

**Leia com ceticismo:** no backtest o modelo acerta ~47-52% dos resultados 1X2, contra ~42% de um palpite fixo. É melhor que chute, mas longe de certeza. Probabilidade de 60% significa que 4 em cada 10 vezes dá outro resultado.

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent / "src"))  # acha o pacote plbig6
import pandas as pd
pd.set_option("display.max_columns", 30)
from plbig6 import data, stats, predict, config
from plbig6.model import DixonColes

In [ ]:
matches = data.season_matches()
train = data.training_set(matches, ['2023-24','2024-25','2025-26'])
model = DixonColes().fit(train, as_of=train['date'].max())
print(f'vantagem de casa: x{__import__("math").exp(model.home_adv):.2f} gols | rho = {model.rho:.3f}')

## Forças atuais (times da temporada)
`ataque` alto = marca mais; `defesa` baixa = sofre menos.

In [ ]:
teams = sorted(set(matches.home))
r = model.ratings().loc[teams].sort_values('forca', ascending=False)
ax = r['forca'].plot.barh(figsize=(7,7)); ax.invert_yaxis(); ax.set_title('Força líquida (ataque - defesa)')
r.round(3)

## Próxima rodada dos Big 6
Se existir `data/escalacoes/rodada_XX.csv`, a escalação entra no cálculo (coluna `escalacao`).

In [ ]:
pend = predict.big6_fixtures(matches)
prox = pend['round'].min()
cols = ['round','date','home','away','escalacao','xg_home','xg_away','p_home','p_draw','p_away','placar_mais_provavel','p_over_2_5','p_btts']
prev = predict.predict_fixtures(model, pend)
prev[prev['round'] == prox][cols].round(3)

## Todos os jogos restantes dos Big 6 até a rodada 38
Previsões de rodadas distantes usam a força de HOJE; elas mudam toda terça com os novos resultados.

In [ ]:
prev[cols].round(3)

## Simulação da temporada (Monte Carlo, 10 mil temporadas)

In [ ]:
sim = predict.simulate_season(model, matches, n_sims=10000)
sim.loc[config.BIG6].sort_values('pontos_medios', ascending=False)

In [ ]:
sim

## Confronto direto (histórico desde 2023-24)

In [ ]:
stats.head_to_head(train, 'Arsenal', 'Man City')